In [14]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np

In [15]:
load_dotenv()
engine = create_engine(os.getenv("SUPABASE_DB_URL"))

query = "SELECT * FROM weather_logs_cleaned ORDER BY timestamp ASC;"
df = pd.read_sql(query, con=engine)
df.tail()

,timestamp,temperature_2m,relative_humidity_2m,rain,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,apparent_temperature,cloud_cover,wind_gusts_10m
105,2026-09-14 10:00:00+00:00,18.37,1.0,0.1,1007.41,0.1,6.12,298.0,13.30,100.0,12.24
106,2026-09-14 11:00:00+00:00,19.17,1.0,0.0,1007.15,0.0,6.48,314.0,14.05,100.0,14.40
107,2026-09-14 12:00:00+00:00,19.77,1.0,0.0,1007.18,0.0,6.12,315.0,14.71,87.0,14.40
108,2026-09-14 13:00:00+00:00,20.92,1.0,0.0,1007.14,0.0,5.40,301.0,15.97,100.0,15.84
109,2026-09-14 14:00:00+00:00,21.62,1.0,0.0,1006.97,0.0,4.68,274.0,16.77,99.0,14.40


In [16]:
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute

# Lag features

In [17]:
df['temp_lag_1'] = df['temperature_2m'].shift(1)
df['temp_lag_3'] = df['temperature_2m'].shift(3)
df['temp_lag_6'] = df['temperature_2m'].shift(6)
df['temp_lag_12'] = df['temperature_2m'].shift(12)
df['temp_lag_24'] = df['temperature_2m'].shift(24)

df['wind_lag_1'] = df['wind_speed_10m'].shift(1)
df['rain_lag_1'] = df['rain'].shift(1)
df['pressure_lag_1'] = df['surface_pressure'].shift(1)

# Rolling Window Features

In [18]:
df['temp_roll_mean_6'] = df['temperature_2m'].rolling(6).mean()
df['temp_roll_mean_24'] = df['temperature_2m'].rolling(24).mean()
df['temp_roll_std_24'] = df['temperature_2m'].rolling(24).std()

df['wind_roll_mean_6'] = df['wind_speed_10m'].rolling(6).mean()
df['rain_roll_sum_24'] = df['rain'].rolling(24).sum()


# Delta Features

In [19]:
df['temp_delta'] = df['temperature_2m'].diff()
df['pressure_delta'] = df['surface_pressure'].diff()
df['wind_delta'] = df['wind_speed_10m'].diff()
df['rain_delta'] = df['rain'].diff()


# Trigonometrical Features

In [20]:
df['wind_dir_sin'] = np.sin(np.deg2rad(df['wind_direction_10m']))
df['wind_dir_cos'] = np.cos(np.deg2rad(df['wind_direction_10m']))


In [21]:
df['cloud_binary'] = (df['cloud_cover'] > 50).astype(int)
df['cloud_roll_mean_6'] = df['cloud_cover'].rolling(6).mean()


In [22]:
df=df.drop("timestamp",axis=1)
df=df.dropna()
df.head()

,temperature_2m,relative_humidity_2m,rain,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,apparent_temperature,cloud_cover,wind_gusts_10m,...,wind_roll_mean_6,rain_roll_sum_24,temp_delta,pressure_delta,wind_delta,rain_delta,wind_dir_sin,wind_dir_cos,cloud_binary,cloud_roll_mean_6
24,17.67,62.0,0.0,1004.02,0.0,9.72,32.0,16.35,87.0,16.56,...,8.22,0.0,-0.40,-0.52,0.36,0.0,0.529919,0.848048,1,86.833333
25,17.52,64.0,0.1,1003.13,0.1,11.16,34.0,16.09,100.0,18.72,...,8.64,0.1,-0.15,-0.89,1.44,0.1,0.559193,0.829038,1,88.000000
26,17.67,65.0,0.0,1003.04,0.0,8.64,23.0,16.72,100.0,19.08,...,8.64,0.1,0.15,-0.09,-2.52,-0.1,0.390731,0.920505,1,88.000000
27,17.72,65.0,0.0,1002.45,0.0,7.20,35.0,16.99,100.0,17.28,...,8.76,0.1,0.05,-0.59,-1.44,0.0,0.573576,0.819152,1,97.500000
28,18.17,60.0,0.0,1002.57,0.0,6.48,53.0,17.32,100.0,12.60,...,8.76,0.1,0.45,0.12,-0.72,0.0,0.798636,0.601815,1,97.833333


In [ ]:
table_name = "weather_logs_features"
df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",  # ha már létezik a tábla, írja felül (futtathatod többször is)
    index=False,           # ne mentse el a pandas indexet külön oszlopként
    method="multi"         # felgyorsítja a nagy mennyiségű adat beszúrását
)